# Serving a model

Training a model is only half the job. This notebook saves a trained model to a checkpoint, starts the project's HTTP inference server inside the notebook, and sends it requests, the same flow the Docker image uses.

In [ ]:
import json
import tempfile
import threading
import urllib.request
from pathlib import Path

from pytorch_examples.serve import build_predictor, make_server

## Train once, save a checkpoint

The first call trains the model and writes the weights. The second call finds the file and loads it without training.

In [ ]:
checkpoint = Path(tempfile.mkdtemp()) / "mlp.pt"

trained = build_predictor(checkpoint, epochs=60)
print("checkpoint written:", checkpoint.exists())

reloaded = build_predictor(checkpoint)
sample = [[0.3, -0.2]]
assert trained.predict(sample) == reloaded.predict(sample)
print("reloaded model gives identical predictions")

## Start the server

`port=0` asks the operating system for a free port. The server runs in a background thread so the notebook stays usable.

In [ ]:
server = make_server(reloaded, "127.0.0.1", 0)
port = server.server_address[1]
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
base_url = f"http://127.0.0.1:{port}"
print("serving on", base_url)

## Call it

In [ ]:
with urllib.request.urlopen(f"{base_url}/health", timeout=10) as response:
    print("health:", json.loads(response.read()))

request = urllib.request.Request(
    f"{base_url}/predict",
    data=json.dumps({"inputs": [[0.1, 0.2], [0.5, -0.5], [-0.6, 0.3]]}).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=10) as response:
    body = json.loads(response.read())

for point, prediction in zip([[0.1, 0.2], [0.5, -0.5], [-0.6, 0.3]], body["predictions"]):
    probs = ", ".join(f"{p:.2f}" for p in prediction["probabilities"])
    print(f"{point} -> class {prediction['class']}  (probabilities: {probs})")

assert len(body["predictions"]) == 3

## Shut down

In [ ]:
server.shutdown()
server.server_close()
thread.join(timeout=5)
print("server stopped")

The same server runs from the command line as `pytorch-examples serve --checkpoint model.pt`, and in a container with `docker run -p 8000:8000 pytorch-examples`. See the deployment page in the docs.